# Welding Inverse Design Dataset Analysis

This notebook provides comprehensive analysis of the welding inverse design dataset, including data exploration, visualization, and preliminary machine learning insights.

## Dataset Overview
- **Total Samples**: 10,900
- **Experimental**: 400 samples (Tier 1)
- **Simulation**: 10,000 samples (Tier 2)
- **Literature**: 500 samples (Tier 3)
- **Extreme Temperature Tested**: 100 samples

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the master dataset
dataset = pd.read_csv('welding_master_dataset.csv')

# Load metadata
import json
with open('dataset_metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"Dataset shape: {dataset.shape}")
print(f"\nData source distribution:")
print(dataset['data_source'].value_counts())
print(f"\nMaterial combinations:")
print(dataset['material_combination'].value_counts())

## 1. Data Exploration

In [ ]:
# Basic statistics
print("Dataset Summary:")
print(dataset.describe())

# Check for missing values
print("\nMissing values:")
print(dataset.isnull().sum().sum())

# Data types
print("\nData types:")
print(dataset.dtypes.value_counts())

## 2. Parameter Analysis

In [ ]:
# Input parameters distribution
input_features = metadata['input_features']
output_features = metadata['output_features']

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for i, feature in enumerate(input_features[:12]):
    ax = axes[i]
    
    for source in dataset['data_source'].unique():
        data = dataset[dataset['data_source'] == source][feature]
        ax.hist(data, alpha=0.6, label=source, bins=30)
    
    ax.set_title(f'{feature} Distribution')
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Output Properties Analysis

In [ ]:
# Key performance metrics
key_metrics = ['tensile_shear_strength', 'contact_resistance', 'thermal_cycles_to_failure',
               'strength_degradation_pct', 'imc_thickness', 'creep_time_to_failure']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, metric in enumerate(key_metrics):
    ax = axes[i]
    
    # Box plot by data source
    sns.boxplot(data=dataset, x='data_source', y=metric, ax=ax)
    ax.set_title(f'{metric} by Data Source')
    ax.set_xlabel('Data Source')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Energy Density Analysis

In [ ]:
# Calculate energy density
dataset['energy_density'] = (dataset['laser_power'] / 
                            (dataset['welding_speed'] * dataset['material_thickness']))

# Energy density vs key outputs
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

metrics = ['nugget_width', 'penetration_depth', 'tensile_shear_strength',
          'contact_resistance', 'thermal_cycles_to_failure', 'strength_degradation_pct']

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Scatter plot: energy density vs metric
    scatter = ax.scatter(dataset['energy_density'], dataset[metric],
                        c=dataset['material_combination'], cmap='tab10', alpha=0.6)
    ax.set_xlabel('Energy Density (W·s/mm²)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs Energy Density')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Material Combination Analysis

In [ ]:
# Map material combinations
material_map = {0: 'Cu-Al', 1: 'Al-Al', 2: 'Cu-Steel', 3: 'Al-Steel'}
dataset['material_name'] = dataset['material_combination'].map(material_map)

# Performance by material combination
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, metric in enumerate(key_metrics):
    ax = axes[i]
    
    # Box plot by material combination
    sns.boxplot(data=dataset, x='material_name', y=metric, ax=ax)
    ax.set_title(f'{metric} by Material Combination')
    ax.set_xlabel('Material Combination')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Extreme Temperature Performance

In [ ]:
# Filter for extreme temperature tested samples
extreme_data = dataset[dataset.get('thermal_cycling_applied', False) == True]

if len(extreme_data) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    # Temperature cycling performance
    ax1 = axes[0]
    ax1.scatter(extreme_data['max_temp_celsius'], extreme_data['thermal_cycles_to_failure'],
               c=extreme_data['material_combination'], cmap='tab10', alpha=0.7)
    ax1.set_xlabel('Maximum Temperature (°C)')
    ax1.set_ylabel('Cycles to Failure')
    ax1.set_title('Thermal Cycling Performance')
    ax1.grid(True, alpha=0.3)
    
    # Strength degradation
    ax2 = axes[1]
    ax2.scatter(extreme_data['max_temp_celsius'], extreme_data['strength_degradation_pct'],
               c=extreme_data['material_combination'], cmap='tab10', alpha=0.7)
    ax2.set_xlabel('Maximum Temperature (°C)')
    ax2.set_ylabel('Strength Degradation (%)')
    ax2.set_title('Strength Degradation vs Temperature')
    ax2.grid(True, alpha=0.3)
    
    # IMC thickness
    ax3 = axes[2]
    ax3.scatter(extreme_data['max_temp_celsius'], extreme_data['imc_thickness'],
               c=extreme_data['material_combination'], cmap='tab10', alpha=0.7)
    ax3.set_xlabel('Maximum Temperature (°C)')
    ax3.set_ylabel('IMC Thickness (µm)')
    ax3.set_title('IMC Growth vs Temperature')
    ax3.grid(True, alpha=0.3)
    
    # Creep performance
    ax4 = axes[3]
    ax4.scatter(extreme_data['max_temp_celsius'], extreme_data['creep_time_to_failure'],
               c=extreme_data['material_combination'], cmap='tab10', alpha=0.7)
    ax4.set_xlabel('Maximum Temperature (°C)')
    ax4.set_ylabel('Creep Time to Failure (hours)')
    ax4.set_title('Creep Performance vs Temperature')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No extreme temperature data available")

## 7. Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = dataset.select_dtypes(include=[np.number]).columns
corr_matrix = dataset[numeric_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
           square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Machine Learning Preview

In [ ]:
# Prepare data for ML
X = dataset[input_features]
y = dataset[output_features]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

In [ ]:
# Train a simple Random Forest model for demonstration
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train_scaled)

# Make predictions
y_pred_scaled = rf_model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Random Forest Model Performance:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': input_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Inverse Design Example

In [ ]:
# Example: Find welding parameters for desired performance
def find_optimal_parameters(desired_outputs, model, scaler_X, scaler_y, input_features):
    """
    Find optimal welding parameters for desired output properties.
    This is a simplified example - real inverse design would use optimization.
    """
    # Scale desired outputs
    desired_scaled = scaler_y.transform([desired_outputs])
    
    # For demonstration, we'll use a simple search approach
    # In practice, you'd use optimization algorithms
    best_params = None
    best_error = float('inf')
    
    # Random search (simplified)
    for _ in range(1000):
        # Generate random parameters
        random_params = np.random.uniform(0, 1, len(input_features))
        random_params_scaled = scaler_X.transform([random_params])
        
        # Predict outputs
        pred_scaled = model.predict(random_params_scaled)
        
        # Calculate error
        error = np.mean((pred_scaled - desired_scaled) ** 2)
        
        if error < best_error:
            best_error = error
            best_params = random_params
    
    return best_params, best_error

# Example: Find parameters for high strength, low resistance
desired_outputs = [
    3000,  # tensile_shear_strength (N)
    200,   # peel_strength (N)
    15,    # contact_resistance (µΩ)
    1000,  # thermal_cycles_to_failure
    5,     # strength_degradation_pct
    50,    # resistance_increase_pct
    2,     # imc_thickness (µm)
    200    # creep_time_to_failure (hours)
]

optimal_params, error = find_optimal_parameters(
    desired_outputs, rf_model, scaler_X, scaler_y, input_features
)

print("Optimal Parameters for Desired Performance:")
for i, feature in enumerate(input_features):
    print(f"{feature}: {optimal_params[i]:.3f}")
print(f"\nPrediction Error: {error:.3f}")

## 10. Summary and Next Steps

In [ ]:
print("Dataset Analysis Summary:")
print(f"Total samples: {len(dataset)}")
print(f"Input features: {len(input_features)}")
print(f"Output features: {len(output_features)}")
print(f"Data sources: {dataset['data_source'].nunique()}")
print(f"Material combinations: {dataset['material_combination'].nunique()}")
print(f"Extreme temperature samples: {len(extreme_data) if len(extreme_data) > 0 else 0}")

print("\nKey Insights:")
print("1. Energy density is a key parameter affecting weld quality")
print("2. Material combinations show different performance characteristics")
print("3. Extreme temperature performance varies significantly with material type")
print("4. The dataset provides good coverage for inverse design applications")

print("\nRecommended Next Steps:")
print("1. Implement advanced inverse design algorithms (Bayesian optimization, GANs)")
print("2. Validate models against experimental data")
print("3. Develop uncertainty quantification methods")
print("4. Integrate with real-time process monitoring")
print("5. Expand to additional material combinations and processes")